# Project 3 Amazon Scraper

### Project Description
Builds an automated extraction and analysis pipeline using Playwright to scrape Amazon search results, and Pandas to clean and analyze the unstructured data to find the best deals.

In [2]:
import asyncio
import pandas as pd
import numpy as np
from playwright.async_api import async_playwright

async def scrape_amazon_products(search_query: str, max_items: int = 10):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.set_extra_http_headers({"User-Agent": "Mozilla/5.0"})
        
        await page.goto(f"https://www.amazon.com/s?k={search_query.replace(' ', '+')}", wait_until="domcontentloaded")
        
        products_data = []
        items = await page.query_selector_all('div[data-component-type="s-search-result"]')
        
        for item in items[:max_items]:
            title_el = await item.query_selector('h2 a span')
            title = await title_el.inner_text() if title_el else "N/A"
            
            price_el = await item.query_selector('span.a-price-whole')
            fract_el = await item.query_selector('span.a-price-fraction')
            price = f"${await price_el.inner_text()}.{await fract_el.inner_text()}" if price_el else "N/A"
            
            rating_el = await item.query_selector('span.a-icon-alt')
            rating = await rating_el.inner_text() if rating_el else "N/A"
            
            if title != "N/A" and price != "N/A":
                products_data.append({"Title": title, "Price": price, "Rating": rating})
                
        await browser.close()
        df = pd.DataFrame(products_data)
        df.to_csv("amazon_raw.csv", index=False)

def analyze_data(file_path: str):
    df = pd.read_csv(file_path)
    df['Price'] = df['Price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df['Rating'] = df['Rating'].str.extract(r'(\d+\.\d+)').astype(float)
    df = df.dropna()
    df['Value_Score'] = np.round(df['Rating'] / df['Price'] * 100, 2)
    print(df.sort_values(by=['Value_Score'], ascending=False).head())

asyncio.run(scrape_amazon_products("mechanical keyboard"))
analyze_data("amazon_raw.csv")

RuntimeError: asyncio.run() cannot be called from a running event loop

### Project Explanation
Uses Playwright's headless Chromium browser to navigate the DOM and extract HTML elements via CSS selectors. Async I/O ensures the CPU isn't blocked. Pandas string methods and regex strip text to float arrays for mathematical 'Value Score' calculations.

### Project Remarks
* **Anti-Bot Defenses:** Real-world usage requires rotating proxies and CAPTCHA solving.
* **Automation:** Can be scheduled via Apache Airflow to power Telegram deal bots.